# 04 Detecting Deviations from Expected Cycling Counts

This notebook identifies deviations from expected cycling traffic during the prediction period from May 2025 to April 2026.

The expected counts were computed in the previous notebook using the final negative binomial model. In this notebook, observed counts are compared with expected counts. A 2-hour interval is classified as a deviation only when the difference is both large in absolute terms and large relative to the expected count.

To avoid classifying unreliable comparisons as deviations, each observation is also required to have at least 10 comparable historical observations in the model-development period. Comparable observations are defined by site, direction, month, weekday and 2-hour interval.

### Packages and path

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

project_folder = Path("..")

processed_folder = project_folder / "data" / "processed"
diagnostics_folder = project_folder / "data" / "diagnostics"

### Loading expected count data

In [ ]:
expected_counts = pd.read_csv(
    processed_folder / "expected_counts.csv"
)

expected_counts["date"] = pd.to_datetime(
    expected_counts["date"],
    errors="coerce"
)

expected_counts.head()

Basic checks

In [ ]:
print("Rows:", expected_counts.shape[0])
print("Columns:", expected_counts.shape[1])

print("Date range:")
print(expected_counts["date"].min(), "to", expected_counts["date"].max())

print("Missing observed counts:", expected_counts["count_rescaled"].isna().sum())
print("Missing expected counts:", expected_counts["expected_count"].isna().sum())

print("Minimum expected count:", expected_counts["expected_count"].min())
print("Maximum expected count:", expected_counts["expected_count"].max())

### Creating deviation variables

In [ ]:
deviation_data = expected_counts.copy()

deviation_data["difference_observed_minus_expected"] = (
    deviation_data["count_rescaled"]
    - deviation_data["expected_count"]
)

deviation_data["relative_difference"] = (
    deviation_data["difference_observed_minus_expected"]
    / deviation_data["expected_count"]
)

### Defining deviations